## Azure Data Engineering Take-Home Assignment
# Project Overview
This project builds an end-to-end data pipeline that extracts data from API endpoints, processes the data using PySpark, and generates business insights such as customer revenue and revenue by state.

This project simulates a real-world data engineering workflow including:

Data ingestion from API \
Data transformation using PySpark \
Data aggregation \
Data export for analytics

The pipeline follows these steps:

1. Extract data from API endpoints
2. Handle pagination to retrieve all records
3. Load data into PySpark DataFrames
4. Transform and join datasets
5. Calculate business metrics
6. Export final dataset

In a production environment, this pipeline could be implemented using:
- Azure Data Factory (orchestration)
- Azure Data Lake (storage)
- Azure Synapse (analytics)

In [1]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

This function fetches data from API endpoints and handles pagination to retrieve all records.

In [2]:
import time

# Create Spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

print("Spark Ready:", spark.version)

Spark Ready: 3.5.0


In [3]:
API_BASE_URL = "http://mock-api:8000"

res = requests.get(f"{API_BASE_URL}/api/v1/health")

print("Status Code:", res.status_code)
print("Response:", res.json())

Status Code: 200
Response: {'status': 'healthy', 'timestamp': '2026-03-25T17:20:24.497794+00:00'}


Authentication (Token)

In [4]:
def get_token():
    res = requests.post(
        f"{API_BASE_URL}/api/v1/auth/token",
        json={
            "username": "candidate",
            "password": "blue-owls-2026"
        }
    )
    
    if res.status_code != 200:
        raise Exception("Auth failed:", res.text)
    
    return res.json()["access_token"]

token = get_token()
print("Token received:", token[:10], "…")

Token received: eyJhbGciOi …


API Fetch Function (Pagination + Retry)

In [5]:
def fetch_all_data(endpoint, retries=3):
    all_data = []
    page = 1
    token = get_token()

    while True:
        attempt = 0

        while attempt < retries:
            headers = {"Authorization": f"Bearer {token}"}
            url = f"{API_BASE_URL}/api/v1/data/{endpoint}?page={page}"

            res = requests.get(url, headers=headers)

            # HANDLE TOKEN EXPIRY
            if res.status_code == 401:
                print(" Token expired. Refreshing...")
                token = get_token()
                continue

            if res.status_code == 200:
                break

            print(f"Retry {attempt+1} for {endpoint} page {page}")
            attempt += 1
            time.sleep(1)

        if res.status_code != 200:
            print(f" Failed: {endpoint}")
            break

        json_data = res.json()
        records = json_data["data"]

        all_data.extend(records)

        print(f"{endpoint} → Page {page} fetched ({len(records)} records)")

        if page >= json_data["pagination"]["total_pages"]:
            break

        page += 1

    print(f" Total {endpoint} records:", len(all_data))
    return all_data

In [6]:
orders = fetch_all_data("orders")
customers = fetch_all_data("customers")
products = fetch_all_data("products")
payments = fetch_all_data("payments")
order_items = fetch_all_data("order_items")

orders → Page 1 fetched (1000 records)
orders → Page 2 fetched (1000 records)
orders → Page 3 fetched (1000 records)
orders → Page 4 fetched (1000 records)
orders → Page 5 fetched (1000 records)
orders → Page 6 fetched (1000 records)
orders → Page 7 fetched (1000 records)
orders → Page 8 fetched (1000 records)
orders → Page 9 fetched (1000 records)
Retry 1 for orders page 10
Retry 2 for orders page 10
orders → Page 10 fetched (1000 records)
Retry 1 for orders page 11
orders → Page 11 fetched (1000 records)
orders → Page 12 fetched (1000 records)
orders → Page 13 fetched (1000 records)
orders → Page 14 fetched (1000 records)
orders → Page 15 fetched (1000 records)
orders → Page 16 fetched (1000 records)
orders → Page 17 fetched (1000 records)
Retry 1 for orders page 18
Retry 2 for orders page 18
orders → Page 18 fetched (1000 records)
orders → Page 19 fetched (1000 records)
orders → Page 20 fetched (1000 records)
orders → Page 21 fetched (1000 records)
Retry 1 for orders page 22
orders 

Fetch All Tables (Bronze Layer Raw Data)

In [7]:
df_orders = spark.createDataFrame(orders)
df_customers = spark.createDataFrame(customers)
df_products = spark.createDataFrame(products)
df_payments = spark.createDataFrame(payments)
df_order_items = spark.createDataFrame(order_items)

df_orders.show(5)

+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+
|         customer_id|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|            order_id|order_purchase_timestamp|order_status|
+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+
|9ef432eb625129730...|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|e481f51cbdc54678b...|     2017-10-02 10:56:33|   delivered|
|b0830fb4747a6c6d2...|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|   delivered|
|41ce2a54c0b03bf34...|2018-08-08 08:55:23|   

Data Cleaning (Silver Layer)

In [8]:
df_payments = df_payments.withColumn(
    "payment_value",
    col("payment_value").cast("double")
)

df_orders = df_orders.withColumn(
    "order_purchase_timestamp",
    col("order_purchase_timestamp").cast("timestamp")
)

Save Bronze Data

In [9]:
df_orders.write.mode("overwrite").parquet("output/bronze/orders")
df_customers.write.mode("overwrite").parquet("output/bronze/customers")
df_products.write.mode("overwrite").parquet("output/bronze/products")
df_payments.write.mode("overwrite").parquet("output/bronze/payments")
df_order_items.write.mode("overwrite").parquet("output/bronze/order_items")

Build Silver Layer

In [10]:
df_orders_clean = df_orders.dropDuplicates()
df_customers_clean = df_customers.dropDuplicates()
df_products_clean = df_products.dropDuplicates()
df_payments_clean = df_payments.dropDuplicates()
df_order_items_clean = df_order_items.dropDuplicates()

Build Gold Layer (Star Schema)

In [11]:
df_fact_sales = df_orders_clean \
    .join(df_order_items_clean, "order_id") \
    .join(df_payments_clean, "order_id")

Dimension Tables

In [12]:
df_dim_customers = df_customers_clean
df_dim_products = df_products_clean

Save Gold Layer

In [13]:
df_fact_sales.write.mode("overwrite").parquet("output/gold/fact_sales")
df_dim_customers.write.mode("overwrite").parquet("output/gold/dim_customers")
df_dim_products.write.mode("overwrite").parquet("output/gold/dim_products")

Revenue by Customer

In [14]:
df_revenue = df_fact_sales.groupBy("customer_id").agg(
    sum("payment_value").alias("total_spent")
)

df_revenue.orderBy("total_spent", ascending=False).show(10)

+--------------------+------------------+
|         customer_id|       total_spent|
+--------------------+------------------+
|728b0d27ba5108af5...|           5527.48|
|b7c889215de76857c...|3974.0400000000004|
|c1474a24d28e0702f...|           3566.88|
|3cecfffcfcdbac8b8...|            3090.0|
|fced842c7dad61e8c...|           2818.74|
|5c9d09439a7815d2c...|            2780.5|
|4b24f5ef0c134fe4d...|           2772.36|
|e5df9583ddbe3f4f3...|           2759.95|
|41b3c5c585822fbde...|           2697.34|
|b1c28f9b72d07054b...|           2681.29|
+--------------------+------------------+
only showing top 10 rows



In [15]:
df_fact_sales.groupBy("product_id").agg(
    sum("payment_value").alias("total_revenue")
).orderBy("total_revenue", ascending=False).show(10)

+--------------------+------------------+
|          product_id|     total_revenue|
+--------------------+------------------+
|7c20caef078d05507...|           5527.48|
|e8316a4667e5870c8...|3974.0400000000004|
|679d1a26cc192f14e...|           3566.88|
|bb50f2e236e5eea01...|           3494.64|
|15151e8a937f6a4d1...|           2818.74|
|8bb27b1d96be90b36...|           2772.36|
|86999f823d7791c57...|           2759.95|
|1ed724b5e4b638eeb...|           2681.29|
|672ffb2231575afa7...|           2281.16|
|25c38557cf793876c...|           2063.27|
+--------------------+------------------+
only showing top 10 rows

